<!--nav--> [🗺 Learning path](README.md) · **8/20** · ◀ [Free Distributed SFT DPO](./Free_Distributed_SFT_DPO.ipynb) · [PostTraining Core Understanding](./PostTraining_Core_Understanding.ipynb) ▶

# Push the Limits: 7B Model Training on Colab Pro A100

Train the **largest competitive model** possible on a single A100 40GB GPU.

| What | Details |
|------|---------|
| **Model** | Qwen2.5-7B (top of 7B class on Open LLM Leaderboard) |
| **Method** | QLoRA — 4-bit base + bf16 adapters on ALL linear layers |
| **Dataset** | UltraChat 200K (the dataset behind Zephyr-7B) |
| **Sequence** | 2048 tokens (4x longer than T4 notebooks) |
| **Benchmark** | 4 Open LLM Leaderboard tasks: TruthfulQA, ARC, HellaSwag, Winogrande |
| **GPU** | A100 40GB (Colab Pro) |

### Why This Setup

```
T4 notebooks (free):    1.1B model, 512 tokens, 1 benchmark
This notebook (A100):   7B model,   2048 tokens, 4 benchmarks
                        ──────────────────────────────────────
                        6.4x bigger model, 4x longer context
```

### Memory Budget (A100 40GB)

```
Qwen2.5-7B in 4-bit (NF4):      ~4.5 GB
LoRA adapters (rank 64):         ~0.3 GB
Gradients + optimizer (bf16):    ~2.0 GB
Activations (seq=2048, batch=4): ~8.0 GB  (with gradient checkpointing)
────────────────────────────────────────
Total:                           ~15 GB  → 25 GB headroom on A100!
```

---
**Runtime:** Colab Pro → Runtime → Change runtime type → A100 GPU

## Step 1: Install & Verify A100

In [ ]:
!pip install -q transformers trl datasets accelerate peft bitsandbytes lm-eval

In [ ]:
import torch, gc, time, os
os.environ["WANDB_DISABLED"] = "true"

assert torch.cuda.is_available(), "No GPU! Change runtime to A100."

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU: %s (%.0f GB)" % (gpu_name, gpu_mem_gb))

if gpu_mem_gb < 30:
    print("WARNING: This notebook is designed for A100 (40GB+).")
    print("You have %.0f GB — training may OOM. Consider using a T4 notebook instead." % gpu_mem_gb)
else:
    print("A100 detected — ready to push limits!")

def gpu_report(label):
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print("%s -> Allocated: %.2f GB | Reserved: %.2f GB" % (label, used, reserved))

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## Step 2: Load Qwen2.5-7B in 4-bit

7.6 billion parameters compressed to ~4.5 GB using NF4 quantization.
The same model in bf16 would need ~15 GB just for weights.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B"
MAX_LENGTH = 2048  # 4x longer than T4 notebooks

# 4-bit quantization — the key to fitting 7B on one GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

gpu_report("7B model loaded in 4-bit")

total_params = sum(p.numel() for p in model.parameters())
print("Total parameters: %.1fB" % (total_params / 1e9))

## Step 3: Apply QLoRA — Adapters on ALL Linear Layers

Rank 64 LoRA on every projection layer = maximum adaptation capacity.
Only ~2% of parameters are trainable, but they touch every layer.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
gpu_report("After applying QLoRA")

## Step 4: Load UltraChat 200K

Multi-turn conversations used to train Zephyr-7B.
We use 10K examples — enough for meaningful training, fast enough for a notebook.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
dataset = dataset.shuffle(seed=42).select(range(10000))

def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text}

formatted = dataset.map(format_chat, remove_columns=dataset.column_names)

print("Training examples: %d" % len(formatted))
print("Max sequence length: %d tokens" % MAX_LENGTH)
print("\nSample (first 200 chars):")
print(formatted[0]["text"][:200])

## Step 5: Train

Pushing A100 hard:
- **Batch size 4** (no gradient accumulation needed)
- **2048 token sequences** (full conversations)
- **Cosine LR** with warmup

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback

class MemoryTracker(TrainerCallback):
    def __init__(self):
        self.peak_mb = 0
        self.losses = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append((state.global_step, logs["loss"]))
        mem = torch.cuda.max_memory_allocated() / 1e6
        if mem > self.peak_mb:
            self.peak_mb = mem

tracker = MemoryTracker()

training_args = SFTConfig(
    output_dir="./qwen7b_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,     # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True,
    gradient_checkpointing=True,
    max_length=MAX_LENGTH,
    dataset_text_field="text",
    logging_steps=10,
    report_to="none",
    save_strategy="no",
    optim="paged_adamw_8bit",          # 8-bit optimizer saves ~50% optimizer memory
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted,
    processing_class=tokenizer,
    callbacks=[tracker],
)

print("Config:")
print("  Model: %s (%.1fB params, QLoRA rank %d)" % (MODEL_NAME, total_params / 1e9, lora_config.r))
print("  Batch: %d x %d grad_accum = %d effective" % (4, 2, 8))
print("  Sequence: %d tokens" % MAX_LENGTH)
print("  Optimizer: paged_adamw_8bit (memory-efficient)")
print("  LR: 2e-4 cosine + 5%% warmup")

In [ ]:
print("=" * 60)
print("  TRAINING: Qwen2.5-7B QLoRA on A100")
print("=" * 60)

start = time.time()
trainer.train()
train_time = time.time() - start

gpu_report("After training")
print("\nPeak GPU: %.1f GB / %.0f GB" % (tracker.peak_mb / 1000, gpu_mem_gb))
print("Training time: %.0f seconds (%.1f minutes)" % (train_time, train_time / 60))

# Save
trainer.save_model("./qwen7b_output/final")
tokenizer.save_pretrained("./qwen7b_output/final")
print("Model saved.")

## Step 6: Quick Generation Test

In [ ]:
model.eval()

test_prompts = [
    "Explain quantum entanglement to a high school student.",
    "Write a Python function that finds the longest palindromic substring.",
    "Is it true that we only use 10% of our brain?",
    "What would happen if the Moon disappeared?",
]

print("=" * 70)
print("  GENERATION SAMPLES (Qwen2.5-7B QLoRA)")
print("=" * 70)

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print("\nQ: %s" % prompt)
    print("A: %s" % resp[:400])
    print("-" * 70)

## Step 7: Benchmark — Open LLM Leaderboard (4 Tasks)

We run the **same 4 benchmarks** used by the Open LLM Leaderboard v1:

| Task | Tests | Examples |
|------|-------|---------|
| **TruthfulQA MC2** | Factuality — avoid common misconceptions | 817 |
| **ARC Easy** | Grade-school science questions | 2376 |
| **HellaSwag** | Commonsense sentence completion | 10042 |
| **Winogrande** | Pronoun resolution / commonsense | 1267 |

Results are directly comparable to published leaderboard scores.

In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM

TASKS = ["truthfulqa_mc2", "arc_easy", "hellaswag", "winogrande"]
TASK_LABELS = {
    "truthfulqa_mc2": "TruthfulQA",
    "arc_easy": "ARC-Easy",
    "hellaswag": "HellaSwag",
    "winogrande": "Winogrande",
}

# Evaluate our fine-tuned model
print("=" * 60)
print("  BENCHMARKING: 4 Open LLM Leaderboard Tasks")
print("=" * 60)

print("\n--- Evaluating FINE-TUNED model ---")
ft_lm = HFLM(pretrained=model, tokenizer=tokenizer)

ft_results = lm_eval.simple_evaluate(
    model=ft_lm,
    tasks=TASKS,
    batch_size=4,
)

ft_scores = {}
for task in TASKS:
    score = ft_results["results"][task].get("acc,none", ft_results["results"][task].get("acc_norm,none", 0))
    ft_scores[task] = score
    print("  %s: %.1f%%" % (TASK_LABELS[task], score * 100))

del ft_lm
clear_gpu()

In [ ]:
# Evaluate BASE model for comparison
print("\n--- Evaluating BASE model (4-bit, no fine-tuning) ---")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_lm = HFLM(pretrained=base_model, tokenizer=tokenizer)

base_results = lm_eval.simple_evaluate(
    model=base_lm,
    tasks=TASKS,
    batch_size=4,
)

base_scores = {}
for task in TASKS:
    score = base_results["results"][task].get("acc,none", base_results["results"][task].get("acc_norm,none", 0))
    base_scores[task] = score
    print("  %s: %.1f%%" % (TASK_LABELS[task], score * 100))

del base_model, base_lm
clear_gpu()

## Step 8: Results

In [ ]:
from IPython.display import HTML, display

# Published leaderboard scores for comparison
leaderboard = {
    "Llama-2-7B":     {"truthfulqa_mc2": 0.389, "arc_easy": 0.765, "hellaswag": 0.760, "winogrande": 0.740},
    "Mistral-7B":     {"truthfulqa_mc2": 0.425, "arc_easy": 0.802, "hellaswag": 0.814, "winogrande": 0.787},
    "Llama-3.1-8B":   {"truthfulqa_mc2": 0.441, "arc_easy": 0.816, "hellaswag": 0.820, "winogrande": 0.774},
}

# Build benchmark comparison table
avg_base = sum(base_scores.values()) / len(base_scores)
avg_ft = sum(ft_scores.values()) / len(ft_scores)

# Header row
header_cells = '<th style="padding:10px;text-align:left;color:#a78bfa;">Model</th>'
for task in TASKS:
    header_cells += '<th style="padding:10px;text-align:center;color:#a78bfa;">%s</th>' % TASK_LABELS[task]
header_cells += '<th style="padding:10px;text-align:center;color:#f0883e;font-weight:700;">Average</th>'

def make_row(name, scores, color, bold=False):
    avg = sum(scores.values()) / len(scores)
    w = "700" if bold else "400"
    bg = "#1a2332" if bold else "transparent"
    row = '<tr style="background:%s;border-bottom:1px solid #21262d;">'\
          '<td style="padding:10px;color:%s;font-weight:%s;">%s</td>' % (bg, color, w, name)
    for task in TASKS:
        row += '<td style="padding:10px;text-align:center;">%.1f%%</td>' % (scores[task] * 100)
    row += '<td style="padding:10px;text-align:center;font-weight:700;color:%s;">%.1f%%</td></tr>' % (color, avg * 100)
    return row

rows = ""
rows += make_row("Qwen2.5-7B (base)", base_scores, "#8b949e")
rows += make_row("Llama-2-7B", leaderboard["Llama-2-7B"], "#6e7681")
rows += make_row("Mistral-7B", leaderboard["Mistral-7B"], "#6e7681")
rows += make_row("Llama-3.1-8B", leaderboard["Llama-3.1-8B"], "#58a6ff")
rows += make_row("OUR Qwen2.5-7B QLoRA", ft_scores, "#3fb950", bold=True)

delta = avg_ft - avg_base
delta_color = "#3fb950" if delta > 0 else "#f85149"
delta_str = "+%.1f%%" % (delta * 100) if delta > 0 else "%.1f%%" % (delta * 100)

html = """
<div style="font-family:-apple-system,sans-serif;max-width:900px;margin:20px 0;">
  <h3 style="color:#c9d1d9;margin-bottom:16px;">Open LLM Leaderboard Comparison (4 Tasks)</h3>
  <table style="width:100%%;border-collapse:collapse;color:#c9d1d9;font-size:13px;">
    <tr style="border-bottom:2px solid #30363d;">%s</tr>
    %s
  </table>
  <div style="margin-top:16px;padding:14px;background:#161b22;border:1px solid #30363d;border-radius:8px;display:flex;gap:32px;">
    <span style="color:#8b949e;">Avg improvement:</span>
    <span style="color:%s;font-weight:700;font-size:18px;">%s</span>
    <span style="color:#8b949e;">|</span>
    <span style="color:#8b949e;">Base avg: %.1f%% → Fine-tuned avg: %.1f%%</span>
  </div>
</div>
""" % (header_cells, rows, delta_color, delta_str, avg_base * 100, avg_ft * 100)

display(HTML(html))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Benchmark comparison (grouped bar)
task_names = [TASK_LABELS[t] for t in TASKS]
base_vals = [base_scores[t] * 100 for t in TASKS]
ft_vals = [ft_scores[t] * 100 for t in TASKS]

x = np.arange(len(task_names))
w = 0.35
axes[0].bar(x - w/2, base_vals, w, label="Base", color="#6e7681", edgecolor="white", linewidth=0.5)
axes[0].bar(x + w/2, ft_vals, w, label="QLoRA FT", color="#3fb950", edgecolor="white", linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(task_names, fontsize=9)
axes[0].set_title("Benchmark Scores (%%)", fontsize=14)
axes[0].set_ylabel("Accuracy (%%)")
axes[0].legend()
axes[0].set_ylim(30, 95)

# Chart 2: Training loss curve
if tracker.losses:
    steps = [x[0] for x in tracker.losses]
    losses = [x[1] for x in tracker.losses]
    axes[1].plot(steps, losses, color="#818cf8", linewidth=2, marker="o", markersize=3)
    axes[1].set_title("Training Loss", fontsize=14)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True, alpha=0.2)

# Chart 3: Comparison with other 7B models (average score)
model_names = ["Llama-2\n7B", "Mistral\n7B", "Llama-3.1\n8B", "Our QLoRA\n7B"]
model_avgs = [
    sum(leaderboard["Llama-2-7B"].values()) / 4 * 100,
    sum(leaderboard["Mistral-7B"].values()) / 4 * 100,
    sum(leaderboard["Llama-3.1-8B"].values()) / 4 * 100,
    avg_ft * 100,
]
colors = ["#6e7681", "#8b949e", "#58a6ff", "#3fb950"]
bars = axes[2].bar(model_names, model_avgs, color=colors, edgecolor="white", linewidth=0.5)
axes[2].set_title("Average Score vs 7B Models", fontsize=14)
axes[2].set_ylabel("Avg Accuracy (%%)")
axes[2].set_ylim(50, 85)
for bar, val in zip(bars, model_avgs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 "%.1f%%" % val, ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

## Summary

In [ ]:
from IPython.display import HTML, display

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

html = """
<div style="font-family:-apple-system,sans-serif;max-width:820px;margin:20px 0;">

  <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px;">
    <div style="background:linear-gradient(135deg,#1a2332,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;text-align:center;">
      <div style="color:#a78bfa;font-size:12px;font-weight:600;">Model</div>
      <div style="color:#f0883e;font-size:22px;font-weight:700;margin:6px 0;">7.6B</div>
      <div style="color:#8b949e;font-size:11px;">Qwen2.5-7B (4-bit)</div>
    </div>
    <div style="background:linear-gradient(135deg,#1a2332,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;text-align:center;">
      <div style="color:#a78bfa;font-size:12px;font-weight:600;">Peak GPU</div>
      <div style="color:#58a6ff;font-size:22px;font-weight:700;margin:6px 0;">%.1f GB</div>
      <div style="color:#8b949e;font-size:11px;">of %.0f GB A100</div>
    </div>
    <div style="background:linear-gradient(135deg,#1a2332,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;text-align:center;">
      <div style="color:#a78bfa;font-size:12px;font-weight:600;">Avg Score</div>
      <div style="color:#3fb950;font-size:22px;font-weight:700;margin:6px 0;">%.1f%%%%</div>
      <div style="color:#8b949e;font-size:11px;">4 benchmarks</div>
    </div>
    <div style="background:linear-gradient(135deg,#1a2332,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;text-align:center;">
      <div style="color:#a78bfa;font-size:12px;font-weight:600;">Time</div>
      <div style="color:#d2a8ff;font-size:22px;font-weight:700;margin:6px 0;">%.0fm</div>
      <div style="color:#8b949e;font-size:11px;">training only</div>
    </div>
  </div>

  <div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:18px;">
    <table style="width:100%%;color:#c9d1d9;font-size:13px;border-spacing:0 8px;">
      <tr><td style="color:#8b949e;">Model</td><td style="text-align:right;font-weight:600;">Qwen2.5-7B</td></tr>
      <tr><td style="color:#8b949e;">Method</td><td style="text-align:right;color:#34d399;">QLoRA (4-bit NF4, rank 64, all layers)</td></tr>
      <tr><td style="color:#8b949e;">Trainable params</td><td style="text-align:right;">%.1fM / %.1fB (%.1f%%%%)</td></tr>
      <tr><td style="color:#8b949e;">Dataset</td><td style="text-align:right;">UltraChat 200K (10K subset)</td></tr>
      <tr><td style="color:#8b949e;">Sequence length</td><td style="text-align:right;">%d tokens</td></tr>
      <tr><td style="color:#8b949e;">Optimizer</td><td style="text-align:right;">PagedAdamW 8-bit</td></tr>
      <tr><td style="color:#8b949e;">GPU</td><td style="text-align:right;">%s (%.0f GB)</td></tr>
    </table>
  </div>
</div>
""" % (
    tracker.peak_mb / 1000, gpu_mem_gb,
    avg_ft * 100,
    train_time / 60,
    trainable / 1e6, total_params / 1e9, 100.0 * trainable / total_params,
    MAX_LENGTH,
    gpu_name, gpu_mem_gb,
)

display(HTML(html))

---

## A100 vs T4 — What You Get with Colab Pro

| | T4 (Free) | A100 (Colab Pro) |
|--|-----------|------------------|
| **VRAM** | 15 GB | 40 GB |
| **Max model (QLoRA)** | ~3B | ~13B |
| **Max model (Full FT)** | ~1.5B | ~3B |
| **Sequence length** | 512 typical | 2048-4096 |
| **Batch size** | 1-2 | 4-8 |
| **bf16 support** | Emulated | Native |
| **Memory bandwidth** | 320 GB/s | 2 TB/s (6x faster) |
| **Training speed** | ~1x | ~3-5x |

### What Made This Possible

```
4-bit quantization (NF4):     7B model → 4.5 GB (was 15 GB in bf16)
LoRA rank 64 on all layers:   Trains 160M params, not 7.6B
PagedAdamW 8-bit:             Halves optimizer memory
Gradient checkpointing:       Recomputes activations, saves ~40%
─────────────────────────────────────────────────────────────────
Result: 7B model trains comfortably in ~15 GB on a 40 GB GPU
```

### Cost Comparison

| Platform | GPU | Cost | This notebook |
|----------|-----|------|---------------|
| Kaggle | T4 | Free | Too small (15 GB) |
| Colab Free | T4 | Free | Too small (15 GB) |
| **Colab Pro** | **A100** | **~$10/mo** | **Fits perfectly** |
| Lambda Labs | A100 | ~$1.10/hr | Works, but costs more |
| RunPod | A100 | ~$1.10/hr | Same |